In [1]:
# Cell 1 (chỉ chạy nếu kernel mới khởi động lại: định nghĩa lại client)
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")

In [2]:
# Cell 2: kiểm tra hạn ở dataset (mặc định) và ở từng bảng/view/MV
# Chỉ đọc thông tin (metadata), không tạo query job nên không tốn tiền.

con_han = 0  # đếm số đối tượng còn hạn, mong đợi cuối cùng = 0

for ds_name in ("lab", "dwh", "raw"):  # mỗi vòng GÁN một tên dataset vào ds_name
    ds = client.get_dataset(f"{PROJECT}.{ds_name}")  # lấy thông tin dataset

    # Hạn mặc định áp cho bảng/partition MỚI tạo trong dataset; None = không có hạn
    print(f"[{ds_name}] default_table_expiration_ms = {ds.default_table_expiration_ms}")
    print(f"[{ds_name}] default_partition_expiration_ms = {ds.default_partition_expiration_ms}")

    for item in client.list_tables(ds):  # vòng lồng: duyệt từng bảng/view/MV trong dataset
        t = client.get_table(item.reference)  # lấy thông tin đầy đủ (list_tables chưa có expires)

        if t.expires is not None:  # có hạn thì in ra và đếm
            con_han += 1
            print(f"   CÒN HẠN: {ds_name}.{t.table_id} ({t.table_type}) hết hạn {t.expires}")

print("Tổng đối tượng còn hạn:", con_han)

[lab] default_table_expiration_ms = None
[lab] default_partition_expiration_ms = None
[dwh] default_table_expiration_ms = None
[dwh] default_partition_expiration_ms = None
[raw] default_table_expiration_ms = None
[raw] default_partition_expiration_ms = None
Tổng đối tượng còn hạn: 0


In [3]:
# Cell 3: kiểm tra riêng bảng partition vừa tạo lại (không chạy query, không tốn tiền)

t = client.get_table(f"{PROJECT}.lab.fact_items_part")

print("Số dòng:", t.num_rows)                      # mong đợi 180771
print("Hạn của bảng (expires):", t.expires)        # mong đợi None
tp = t.time_partitioning                           # cấu hình partition của bảng
print("Partition theo:", tp.type_, "| cột:", tp.field)
print("Hạn của partition (expiration_ms):", tp.expiration_ms)  # mong đợi None
print("require_partition_filter:", t.require_partition_filter)

Số dòng: 180771
Hạn của bảng (expires): None
Partition theo: MONTH | cột: order_date
Hạn của partition (expiration_ms): None
require_partition_filter: True
